In [1]:
import numpy as np
import pandas as pd
import sys
import json
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

In [2]:
from src.pipeline.config import (
    TRAIN_DF_PATH,
	RANDOM_SEED
)

In [3]:
df = pd.read_parquet(TRAIN_DF_PATH, engine="pyarrow")

In [4]:
df.head()

,TransactionID,isFraud,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,TransactionAmtMax,TransactionAmtMin,TransactionAmtStd,TransactionAmt_Z,TransactionAmt_to_Mean,DayOfWeek,HourSin,HourCos,DayOfWeekSin,DayOfWeekCos
0,2987000,0,68.5,W,13926,-1.0,150.0,discover,142.0,credit,...,317.50,68.5,176.069589,-0.707107,0.354922,1,0.0,1.0,0.781831,0.62349
1,2987001,0,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,...,4592.02,29.0,614.242501,-0.407218,0.103894,1,0.0,1.0,0.781831,0.62349
2,2987002,0,59.0,W,4663,490.0,150.0,visa,166.0,debit,...,122.99,25.0,22.055991,0.297718,1.125234,1,0.0,1.0,0.781831,0.62349
3,2987003,0,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,...,3190.00,6.0,248.958879,-0.301440,0.399852,1,0.0,1.0,0.781831,0.62349
4,2987004,0,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,...,50.00,50.0,0.000000,0.000000,1.000000,1,0.0,1.0,0.781831,0.62349


In [5]:
X = df.drop(["DeviceInfo", "isFraud", "uid", "DayOfWeek", "TransactionID"], axis=1)
y = df["isFraud"]

In [6]:
kaggle_cat_cols = (
    ['ProductCD'] +
    [f'card{i}' for i in range(1, 7)] +
    ['addr1', 'addr2'] +
    ['P_emaildomain', 'R_emaildomain'] +
    [f'M{i}' for i in range(1, 10)] +
    ['DeviceType', 'DeviceInfo'] +
    [f'id_{i}' for i in range(12, 39)]
)

cat_features = [col for col in kaggle_cat_cols if col in X.columns]

for col in cat_features:
    X[col] = X[col].fillna('missing').astype(str)

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

In [8]:
from catboost import CatBoostClassifier

clf = CatBoostClassifier(
    iterations=7000,
    learning_rate=0.05,
    depth=7,
    l2_leaf_reg=5,
    max_ctr_complexity=3,
    eval_metric='AUC',
    custom_metric=['Logloss'],
    random_seed=RANDOM_SEED,
    early_stopping_rounds=150,
    task_type='GPU',
    verbose=50
)

In [9]:
clf.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    cat_features=cat_features,
    use_best_model=True
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7044275	best: 0.7044275 (0)	total: 277ms	remaining: 32m 16s
50:	test: 0.8931346	best: 0.8931346 (50)	total: 9.04s	remaining: 20m 32s
100:	test: 0.9173688	best: 0.9173688 (100)	total: 16.7s	remaining: 19m 2s
150:	test: 0.9285533	best: 0.9285533 (150)	total: 24.4s	remaining: 18m 27s
200:	test: 0.9352304	best: 0.9352304 (200)	total: 32.5s	remaining: 18m 20s
250:	test: 0.9395629	best: 0.9395629 (250)	total: 40.8s	remaining: 18m 17s
300:	test: 0.9426914	best: 0.9426914 (300)	total: 48.8s	remaining: 18m 6s
350:	test: 0.9443084	best: 0.9443139 (349)	total: 57.1s	remaining: 18m 2s
400:	test: 0.9457520	best: 0.9457548 (399)	total: 1m 5s	remaining: 17m 55s
450:	test: 0.9467402	best: 0.9467402 (450)	total: 1m 13s	remaining: 17m 44s
500:	test: 0.9484901	best: 0.9484901 (500)	total: 1m 21s	remaining: 17m 34s
550:	test: 0.9491217	best: 0.9491217 (550)	total: 1m 29s	remaining: 17m 26s
600:	test: 0.9496402	best: 0.9496402 (600)	total: 1m 38s	remaining: 17m 24s
650:	test: 0.9505771	best: 0.9

CatBoostClassifier(custom_metric=['Logloss'], depth=7, early_stopping_rounds=150, eval_metric='AUC', iterations=7000, l2_leaf_reg=5, learning_rate=0.05, max_ctr_complexity=3, random_seed=42, task_type='GPU', verbose=50)

In [10]:
from sklearn.metrics import roc_auc_score

val_preds = clf.predict_proba(X_test)[:, 1]

auc_score = roc_auc_score(y_test, val_preds)
print(f"Baseline ROC-AUC: {auc_score:.5f}")

Baseline ROC-AUC: 0.97089


In [11]:
feature_imp = pd.Series(clf.get_feature_importance(), index=X_train.columns)
strong_features = feature_imp[feature_imp > 0.01].index.tolist()
weak_features = feature_imp[feature_imp <= 0.01].index.tolist()

print(feature_imp.sort_values(ascending=False).head(10), "\n")
print(f"Strong features: {len(strong_features)}")
print(f"Weak features: {len(weak_features)}")

card1                5.829012
C13                  5.737009
C1                   3.973865
M5                   3.499580
card2                3.093179
TransactionAmtMax    2.772158
addr1                2.531663
P_emaildomain        2.459315
TransactionAmt       2.343953
TransactionAmtMin    2.319792
dtype: float64 

Strong features: 270
Weak features: 32


In [12]:
print(*weak_features)

V1 V14 V27 V41 V107 V110 V113 V117 V118 V119 V120 V121 V122 V138 V142 V146 V174 V181 V183 V185 V191 V194 V195 V197 V226 V227 V241 V252 V305 V334 V336 id_04


In [14]:
clf.save_model("../models/catboost_fraud_model_v1.cbm")

In [15]:
cat_cols_selected = [col for col in cat_features if col in strong_features]

metadata = {
    "features": strong_features,
    "cat_features": cat_cols_selected,
    "best_auc": float(clf.get_best_score()['validation']['AUC'])
}

In [17]:
with open("../models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)